In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("C:\\Users\\delta\\Predictive-maintenence-iot\\dataset\\ai4i2020.csv")
y = df["Machine failure"]
X = df.drop(columns=["UDI","Product ID","Machine failure"])
X = pd.get_dummies(X,columns=["Type"],drop_first=True)
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.20,random_state=42,stratify=y)
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)
failure_probability = pipeline.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test,failure_probability)
print("Threshold Values :", len(thresholds))
print("Precision Values :", len(precision))
print("Recall Values :", len(recall))

In [ ]:
threshold_df = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": precision[:-1],
    "Recall": recall[:-1]
})

print("Threshold Analysis Table")
threshold_df.head(10)

In [ ]:
print("Threshold Statistics")
print("="*50)
print(threshold_df.describe())

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds,precision[:-1],linewidth=2,label="Precision")
plt.xlabel("Decision Threshold")
plt.ylabel("Precision")
plt.title("Threshold vs Precision")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds,recall[:-1],linewidth=2,color="green",label="Recall")
plt.xlabel("Decision Threshold")
plt.ylabel("Recall")
plt.title("Threshold vs Recall")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds,precision[:-1],label="Precision")
plt.plot(thresholds,recall[:-1],label="Recall")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision vs Recall")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
f1_scores = (2 *precision[:-1] *recall[:-1]) / (precision[:-1] +recall[:-1] +1e-10)
threshold_df["F1 Score"] = f1_scores
threshold_df.head()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds,f1_scores,linewidth=2,color="red",label="F1 Score")
plt.xlabel("Decision Threshold")
plt.ylabel("F1 Score")
plt.title("Threshold vs F1 Score")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds,precision[:-1],label="Precision",linewidth=2)
plt.plot(thresholds,recall[:-1],label="Recall",linewidth=2)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold vs Precision and Recall")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_index = np.argmax(f1_scores)
best_threshold = thresholds[best_index]
best_precision = precision[best_index]
best_recall = recall[best_index]
best_f1 = f1_scores[best_index]
print("Best Threshold :", round(best_threshold,3))
print("Precision      :", round(best_precision,3))
print("Recall         :", round(best_recall,3))
print("F1 Score       :", round(best_f1,3))

In [ ]:
best_result = pd.DataFrame({
    "Metric": [
        "Best Threshold",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        best_threshold,
        best_precision,
        best_recall,
        best_f1
    ]})
best_result

In [ ]:
top_thresholds = threshold_df.sort_values(by="F1 Score",ascending=False)
top_thresholds.head(10)

In [ ]:
high_precision = threshold_df[threshold_df["Precision"] >= 0.90]
print(high_precision.head(10))

In [ ]:
high_recall = threshold_df[threshold_df["Recall"] >= 0.90]
print(high_recall.head(10))

In [ ]:
threshold_df["Category"] = pd.cut(
    threshold_df["Threshold"],
    bins=[0,0.25,0.50,0.75,1],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High"
    ]
)
threshold_df.head()

In [ ]:
summary = threshold_df.groupby("Category")[[
    "Precision",
    "Recall",
    "F1 Score"
]].mean()
summary

In [ ]:
threshold_df.to_csv("threshold_analysis.csv",index=False)
print("Threshold analysis exported successfully.")

In [ ]:
report = f"""
==============================
Week 4 Day 3 Report
==============================

Task Completed
--------------------------
✓ Generated Threshold vs Precision plot
✓ Generated Threshold vs Recall plot
✓ Compared Precision and Recall
✓ Calculated F1 Score at each threshold
✓ Identified optimal decision threshold

Best Threshold : {best_threshold:.3f}
Precision      : {best_precision:.3f}
Recall         : {best_recall:.3f}
F1 Score       : {best_f1:.3f}

Conclusion
--------------------------
The threshold analysis shows how changing the decision threshold affects model performance. The selected threshold provides a balanced trade-off between Precision and Recall, helping reduce false alarms while still identifying most machine failures.
"""

print(report)